# 15.5 因子分解机 FM & 场感知 FFM / Factorization Machines & Field-aware FM

**中文**：前面的 MF 只有"用户 ID × 物品 ID"两种特征。但真实 CTR（点击率）预估里，一条样本有几十上百个特征：用户性别、年龄、城市、设备、物品类别、价格、时段……而且**真正的信号往往藏在特征的"组合"里**——比如"男性 × 动作片"点击率高、"女性 × 爱情片"点击率高。**因子分解机（FM）** 就是把 MF 的"隐向量点积"思想推广到**任意特征对之间**，是工业 CTR 预估的奠基模型。
**English**: MF used only two features ("user ID × item ID"). But in real CTR (click-through-rate) prediction, each sample has dozens-to-hundreds of features: gender, age, city, device, item category, price, hour… and **the real signal often lives in feature *combinations*** — e.g. "male × action movie" clicks high, "female × romance" clicks high. **Factorization Machines (FM)** generalize MF's "latent-vector dot product" to **every pair of features**, the foundational model of industrial CTR prediction.

---

**中文**：FM 的预测公式：
**English**: The FM prediction formula:

$$\hat y(\mathbf{x}) = w_0 + \sum_{i=1}^{F} w_i x_i + \sum_{i=1}^{F}\sum_{j=i+1}^{F} \langle \mathbf{v}_i,\mathbf{v}_j\rangle\, x_i x_j$$

**中文**：逐项看：$w_0$ 全局偏置；$\sum w_i x_i$ 是**线性部分**（就是逻辑回归 LR）；第三项是**二阶特征交叉**——任意两个特征 $i,j$ 的交互强度，用它们各自隐向量的点积 $\langle\mathbf{v}_i,\mathbf{v}_j\rangle$ 来表示。$x_i$ 是特征值（类别特征 one-hot 后是 0/1）。
**English**: Term by term: $w_0$ global bias; $\sum w_i x_i$ is the **linear part** (exactly logistic regression, LR); the third term is **second-order feature crossing** — the interaction strength of any two features $i,j$, expressed by the dot product $\langle\mathbf{v}_i,\mathbf{v}_j\rangle$ of their latent vectors. $x_i$ is the feature value (0/1 after one-hot for categoricals).

**中文**：**为什么不直接给每个特征对学一个权重 $w_{ij}$？** 因为特征对数量是 $O(F^2)$，而且绝大多数特征对在训练集里**从未共现**（比如"用户A × 物品B"只出现一次甚至零次），无法学到可靠权重。FM 的天才之处：用**隐向量点积**代替独立权重，让"用户A × 物品B"的交互可以从"用户A × 物品C""用户D × 物品B"等**间接学到**——这就是因子分解对抗稀疏的力量。
**English**: **Why not learn an independent weight $w_{ij}$ per feature pair?** Because there are $O(F^2)$ pairs and most pairs **never co-occur** in training (e.g. "user A × item B" appears once or never), so their weights can't be learned reliably. FM's genius: replace independent weights with a **latent-vector dot product**, so "user A × item B" is learned **indirectly** from "user A × item C," "user D × item B," etc. — factorization's power against sparsity.

**中文**：直接算所有特征对是 $O(F^2)$，但有一个**著名恒等式**把它降到 $O(kF)$（$k$ 是隐向量维度）：
**English**: Computing all pairs is $O(F^2)$, but a **famous identity** reduces it to $O(kF)$ ($k$ = latent dim):

$$\sum_{i<j}\langle\mathbf{v}_i,\mathbf{v}_j\rangle x_i x_j = \frac{1}{2}\sum_{f=1}^{k}\Big[\big(\sum_i v_{if}x_i\big)^2 - \sum_i v_{if}^2 x_i^2\Big]$$

**中文**：因为样本里非零特征很少（稀疏），实际只需遍历**非零特征**，所以 FM 在亿级特征上依然高效。
**English**: Since a sample has few nonzero features (sparse), we only loop over **nonzero features**, so FM stays efficient even with billions of features.

> 💡 **面试速查 / Interview cheat-sheet（★★★ CTR 必考）**
> **中文**：**FM = LR + 二阶交叉(用隐向量点积参数化)**。优点：① 自动学特征交叉，省去人工特征工程；② 用因子化对抗稀疏（未共现的特征对也能泛化）；③ 线性复杂度 $O(kF)$。**FM vs MF**：MF 是 FM 只保留 user/item 两个 ID 特征的特例。**FFM（场感知 FM）**：每个特征对**不同的场(field)**用**不同的隐向量** $\mathbf{v}_{i,f_j}$——交互 $\langle\mathbf{v}_{i,f_j},\mathbf{v}_{j,f_i}\rangle$；表达力更强（Criteo/Avazu 比赛冠军方案），但参数量 ×场数、易过拟合、训练更慢。
> **English**: **FM = LR + 2nd-order crossing (parameterized by latent dot products)**. Pros: ① auto-learns crosses (less manual feature engineering); ② factorization fights sparsity (generalizes to unseen pairs); ③ linear complexity $O(kF)$. **FM vs MF**: MF is the special case of FM with only user/item ID features. **FFM (field-aware FM)**: each feature has a **separate latent vector per field** $\mathbf{v}_{i,f_j}$ — interaction $\langle\mathbf{v}_{i,f_j},\mathbf{v}_{j,f_i}\rangle$; more expressive (Criteo/Avazu competition winners) but params × #fields, prone to overfitting, slower.


In [ ]:

# ============================================================
# 实验一：合成"纯交叉信号"数据，证明 FM 的机制 / Synthetic interaction-only data
# 中文：我们故意造一个"信号只在特征组合里"的数据：男性+动作 / 女性+爱情 才高点击，
#       而单看性别或单看类型都无法预测——这正是线性模型(LR)无能为力、FM 大显身手的场景。
# English: We deliberately build data where signal lives ONLY in the combination:
#          male+Action / female+Romance click high, while gender alone or genre alone can't predict.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(0)
genders=["Male","Female"]; genres=["Action","Romance","Comedy","Horror"]
N=8000
g  = rng.integers(0,2,N)                              # 性别 0=M,1=F / gender
gr = rng.integers(0,4,N)                              # 类型 0..3 / genre
# 点击 logit：只有 (男,动作) 和 (女,爱情) 两个组合显著加分 —— 纯交叉
logit = -1.0 + 2.6*((g==0)&(gr==0)) + 2.6*((g==1)&(gr==1))
p = 1/(1+np.exp(-logit)); y=(rng.random(N)<p).astype(float)   # 采样点击标签 / sample clicks

# 特征编码：6 个特征(性别2 + 类型4)，每条样本 2 个非零特征 / encode: 6 feats, 2 active per sample
# 特征下标：0,1=性别；2,3,4,5=类型。场(field)：性别=0, 类型=1
F=6
samples=[(np.array([g[i], 2+gr[i]]), np.array([0,1])) for i in range(N)]   # (idx, field) per sample
tr=range(0,6000); te=range(6000,N)                    # 训练/测试切分 / split
print(f"样本数 N={N}, 特征数 F={F}, 整体点击率 CTR={y.mean():.2%}")
# 看看真实信号：每个(性别,类型)组合的点击率 / true CTR per combo
print("真实组合点击率 / true CTR by (gender,genre):")
for a in range(2):
    print("  ",genders[a],{genres[b]:round(y[(g==a)&(gr==b)].mean(),2) for b in range(4)})


**中文**：现在从零实现三个模型：**LR**（只有线性项）、**FM**（线性 + 二阶交叉，用上面的 $O(kF)$ 技巧）、**FFM**（场感知）。都用逻辑损失 + SGD。对比它们在这份"纯交叉"数据上的 AUC，直接看交叉建模带来多大提升。
**English**: Now implement three models from scratch: **LR** (linear only), **FM** (linear + 2nd-order crossing with the $O(kF)$ trick), **FFM** (field-aware). All use logistic loss + SGD. Compare their AUC on this interaction-only data to see directly how much crossing helps.


In [ ]:

# ============================================================
# 从零实现 LR / FM / FFM（逻辑损失 SGD）/ LR / FM / FFM from scratch
# ============================================================
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-30,30)))
def auc(yy,pp):                                        # 手写 AUC = 正负样本对中排序正确的比例
    o=np.argsort(pp); r=np.empty(len(pp)); r[o]=np.arange(len(pp))
    npos=yy.sum(); nneg=len(yy)-npos
    return (r[yy==1].sum()-npos*(npos-1)/2)/(npos*nneg)

def train_model(model="lr", k=8, epochs=25, lr=0.1, reg=1e-5):
    rng=np.random.default_rng(1)
    w=np.zeros(F); b=0.0
    V =rng.normal(0,0.1,(F,k))        # FM 隐向量 / FM latent vectors
    Vff=rng.normal(0,0.1,(F,2,k))     # FFM：每个特征对每个场一个隐向量 / per (feature, field)
    for ep in range(epochs):
        for t in rng.permutation(6000):
            idx,fld=samples[t]; lin=b+w[idx].sum()
            if model=="lr":
                pred=lin
            elif model=="fm":
                Vi=V[idx]; s=Vi.sum(0)
                pred=lin+0.5*(s@s-(Vi*Vi).sum())              # O(kF) 交叉项 / crossing
            else:  # ffm：显式遍历特征对，用"对方场"的隐向量 / explicit pairs, field-aware
                inter=0.0
                for a in range(len(idx)):
                    for c in range(a+1,len(idx)):
                        inter+=Vff[idx[a],fld[c]]@Vff[idx[c],fld[a]]
                pred=lin+inter
            gd=sigmoid(pred)-y[t]                              # 逻辑损失梯度 / logistic grad
            b-=lr*gd; w[idx]-=lr*(gd+reg*w[idx])
            if model=="fm":
                V[idx]-=lr*(gd*(s-Vi)+reg*Vi)                 # dV_i = g*(s - V_i)
            elif model=="ffm":
                for a in range(len(idx)):
                    for c in range(a+1,len(idx)):
                        va=Vff[idx[a],fld[c]].copy(); vb=Vff[idx[c],fld[a]].copy()
                        Vff[idx[a],fld[c]]-=lr*(gd*vb+reg*va)
                        Vff[idx[c],fld[a]]-=lr*(gd*va+reg*vb)
    return w,b,V,Vff

def predict(model,w,b,V,Vff,idx,fld):
    lin=b+w[idx].sum()
    if model=="lr": return sigmoid(lin)
    if model=="fm":
        Vi=V[idx]; s=Vi.sum(0); return sigmoid(lin+0.5*(s@s-(Vi*Vi).sum()))
    it=sum(Vff[idx[a],fld[c]]@Vff[idx[c],fld[a]] for a in range(len(idx)) for c in range(a+1,len(idx)))
    return sigmoid(lin+it)

res={}
for mdl in ["lr","fm","ffm"]:
    w,b,V,Vff=train_model(mdl)
    pr=np.array([predict(mdl,w,b,V,Vff,*samples[i]) for i in te])
    res[mdl]=(auc(y[list(te)],pr),(w,b,V,Vff))
    print(f"{mdl.upper():4}  test AUC = {res[mdl][0]:.4f}")


**中文**：结果清晰：**LR ≪ FM ≈ FFM**。LR 只能学"动作片整体点击率""男性整体点击率"这类**主效应**，无法表达"男性**且**动作"这种交叉，所以 AUC 明显偏低；FM 用隐向量点积捕捉到了交叉，AUC 大幅跃升。**诚实地说**：这里 FFM 并没有超过 FM（两者基本持平）——因为本例只有 **2 个场**、交叉关系也简单，FFM 多出来的"每场一套隐向量"没有用武之地，反而徒增参数。FFM 的优势要在**很多场、且不同场对交叉作用差异很大**时才体现（如 Criteo 有几十个场）。
**English**: Clear result: **LR ≪ FM ≈ FFM**. LR can only learn **main effects** ("Action's overall CTR," "males' overall CTR") but not the cross "male **AND** Action," so its AUC is clearly low; FM captures the cross via latent dot products, lifting AUC sharply. **Honestly**: FFM does *not* beat FM here (they essentially tie) — with only **2 fields** and a simple interaction, FFM's extra "one latent vector per field" buys nothing and merely adds parameters. FFM's edge only appears with **many fields whose interactions differ a lot across field pairs** (e.g. Criteo's dozens of fields).

**中文**：最直观的证据是把每个模型学到的"(性别×类型)点击率"画成热力图，和真实规则对比。FM/FFM 应该能重建出"棋盘格"模式（对角两格高），而 LR 只能给出"行列可加"的平滑模式。
**English**: The most intuitive evidence: plot each model's predicted "(gender × genre) CTR" as a heatmap against the true rule. FM/FFM should reconstruct the "checkerboard" (two diagonal cells high), while LR can only produce an "additive (row+column)" smooth pattern.


In [ ]:

# ============================================================
# 可视化：真实 vs LR vs FM 的 (性别×类型) 点击率热力图 / heatmaps
# ============================================================
def grid(model):
    w,b,V,Vff=res[model][1]; M=np.zeros((2,4))
    for a in range(2):
        for c in range(4):
            M[a,c]=predict(model,w,b,V,Vff,np.array([a,2+c]),np.array([0,1]))
    return M
true_grid=np.array([[y[(g==a)&(gr==c)].mean() for c in range(4)] for a in range(2)])

fig,ax=plt.subplots(1,3,figsize=(14,3.4))
for axi,(name,M) in zip(ax,[("真实 / True",true_grid),("LR (主效应)",grid("lr")),("FM (交叉)",grid("fm"))]):
    im=axi.imshow(M,cmap="YlOrRd",vmin=0,vmax=1,aspect="auto")
    axi.set_xticks(range(4)); axi.set_xticklabels(genres,rotation=30); axi.set_yticks(range(2)); axi.set_yticklabels(genders)
    for a in range(2):
        for c in range(4): axi.text(c,a,f"{M[a,c]:.2f}",ha="center",va="center",fontsize=9)
    axi.set_title(name)
plt.colorbar(im,ax=ax,fraction=0.025); plt.suptitle("(性别×类型) 点击率：FM 重建棋盘格，LR 只能可加 / FM recovers checkerboard, LR can't")
plt.savefig("/tmp/rec05_viz.png",dpi=80); plt.show()
print("LR AUC %.3f | FM AUC %.3f | FFM AUC %.3f"%(res["lr"][0],res["fm"][0],res["ffm"][0]))


**中文**：合成数据证明了机制。但**机制有用 ≠ 在任何真实数据上都赢**。下面把 FM 放到真实的 MovieLens-CTR 任务上：用 用户ID、物品ID、性别、年龄段、职业、电影类型、年代 这些特征预测"是否打高分(≥4)"，诚实地看 FM 能否打败 LR。
**English**: The synthetic data proves the mechanism. But **a useful mechanism ≠ winning on every real dataset**. Now put FM on a real MovieLens-CTR task: predict "rated highly (≥4)?" from user ID, item ID, gender, age bucket, occupation, genres, decade — and honestly check whether FM beats LR.


In [ ]:

# ============================================================
# 实验二：真实 MovieLens-CTR / Real MovieLens CTR task
# ============================================================
import os, pandas as pd
R_DIR=os.path.expanduser("~/.cache/dsfs_recsys/ml-100k")
rat=pd.read_csv(os.path.join(R_DIR,"u.data"),sep="\t",names=["user","item","rating","ts"])
usr=pd.read_csv(os.path.join(R_DIR,"u.user"),sep="|",names=["user","age","gender","occ","zip"])
GEN=["unknown","Action","Adventure","Animation","Children","Comedy","Crime","Documentary","Drama",
     "Fantasy","FilmNoir","Horror","Musical","Mystery","Romance","SciFi","Thriller","War","Western"]
mv=pd.read_csv(os.path.join(R_DIR,"u.item"),sep="|",encoding="latin-1",header=None,
               names=["item","title","date","v","url"]+GEN)
mv["decade"]=((mv["date"].str.extract(r"(\d{4})").astype(float).fillna(1990)//10)*10).astype(int).astype(str)
d=rat.merge(usr,on="user").merge(mv,on="item").sort_values("ts")
d["label"]=(d["rating"]>=4).astype(int)
d["agebkt"]=pd.cut(d["age"],[0,18,25,35,45,56,100],labels=False).astype(str)

# 构造特征(field, value)列表 / build (field,value) feature lists
def rows_of(df):
    base=df[["user","item","gender","agebkt","occ","decade"]].astype(str)
    out=[]
    for (u,i,gd,ab,oc,dc),gvec in zip(base.itertuples(index=False,name=None), df[GEN].values):
        f=[("user",u),("item",i),("gender",gd),("age",ab),("occ",oc),("decade",dc)]
        f+= [("genre",GEN[j]) for j in range(len(GEN)) if gvec[j]==1]
        out.append(f)
    return out
cut=int(len(d)*0.8); rtr=rows_of(d.iloc[:cut]); rte=rows_of(d.iloc[cut:])
ytr=d["label"].values[:cut].astype(float); yte2=d["label"].values[cut:].astype(float)

# 从训练集建立特征索引 + 场 / index features and fields from train
fidx={}; ffield=[]
for row in rtr:
    for fv in row:
        if fv not in fidx: fidx[fv]=len(fidx); ffield.append(fv[0])
flds=sorted(set(ffield)); fld2=  {f:i for i,f in enumerate(flds)}
Fr=len(fidx)
def enc(rows):
    out=[]
    for row in rows:
        idx=[fidx[fv] for fv in row if fv in fidx]
        out.append(np.array(idx))
    return out
Xtr=enc(rtr); Xte=enc(rte)
print(f"真实任务：特征数 F={Fr}, 场数 fields={len(flds)}, 训练样本 {len(Xtr)}, 正例率 {ytr.mean():.2%}")


In [ ]:

# ============================================================
# 在真实任务上训练 LR vs FM（复用 O(kF) 技巧）/ train LR vs FM on real task
# ============================================================
def train_real(model="fm", k=8, epochs=8, lr=0.05, reg=1e-5):
    rng=np.random.default_rng(0); w=np.zeros(Fr); b=0.0; V=rng.normal(0,0.05,(Fr,k))
    for ep in range(epochs):
        for t in rng.permutation(len(Xtr)):
            idx=Xtr[t]; lin=b+w[idx].sum()
            if model=="lr": pred=lin
            else:
                Vi=V[idx]; s=Vi.sum(0); pred=lin+0.5*(s@s-(Vi*Vi).sum())
            gd=sigmoid(pred)-ytr[t]; b-=lr*gd; w[idx]-=lr*(gd+reg*w[idx])
            if model=="fm": V[idx]-=lr*(gd*(s-Vi)+reg*Vi)
    return w,b,V
def pred_real(model,w,b,V,idx):
    lin=b+w[idx].sum()
    if model=="lr": return sigmoid(lin)
    Vi=V[idx]; s=Vi.sum(0); return sigmoid(lin+0.5*(s@s-(Vi*Vi).sum()))

wl,bl,_=train_real("lr"); pl=np.array([pred_real("lr",wl,bl,None,idx) for idx in Xte])
wf,bf,Vf=train_real("fm"); pf=np.array([pred_real("fm",wf,bf,Vf,idx) for idx in Xte])
print(f"{'方法/method':<26}{'AUC':>8}")
print(f"{'LR (线性, no cross)':<26}{auc(yte2,pl):>8.4f}")
print(f"{'FM (二阶交叉)':<26}{auc(yte2,pf):>8.4f}")


**中文**：诚实结果——在真实 MovieLens-CTR 上，**FM 与 LR 几乎打平**（AUC 都在 ~0.68 附近），FM 并没有像合成数据上那样碾压。这不是 FM 没用，而是一个非常重要、面试能加分的洞察：
**English**: Honest result — on real MovieLens-CTR, **FM and LR are nearly tied** (both AUC ~0.68); FM does *not* dominate as on synthetic data. This is not "FM is useless" but a crucial, interview-worthy insight:

**中文**：
1. **交叉建模只有在"交叉里真有信号、且线性项没吃掉它"时才有增益**。这里主导特征是高基数的 user_id / item_id，它们的**线性项**（$w_{\text{user}}$、$w_{\text{item}}$）已经吸收了"用户偏好严格度""电影整体受欢迎度"这些主效应；而 ID×ID 的交叉在**时间外推**（测试集有新电影/口味漂移）下容易过拟合、泛化有限。
2. **合成实验的价值**：它剥离了一切干扰，纯净地证明 FM 能学交叉 LR 不能——这是理解机制的正确方式。真实数据则提醒我们：**模型增益依赖数据中交叉信号的强弱**，不能想当然。
3. **这正是后续 DeepFM/DCN（15.7）的动机**：单靠二阶交叉不够，需要**深层网络学高阶非线性交叉** + **保留低阶记忆**，才能在工业 CTR 上稳定超越 LR/FM。

**English**:
1. **Crossing helps only when the cross truly carries signal that the linear part hasn't absorbed**. Here the dominant features are high-cardinality user_id / item_id, whose **linear terms** ($w_{\text{user}},w_{\text{item}}$) already soak up main effects (user strictness, movie popularity); the ID×ID cross tends to overfit under **temporal extrapolation** (new movies / taste drift in the test set), so it generalizes little.
2. **Why the synthetic experiment matters**: it strips away confounds and cleanly proves FM learns crosses LR cannot — the right way to understand the mechanism. Real data then reminds us: **the gain depends on how strong the cross signal is**, never assume.
3. **This motivates DeepFM/DCN (15.7)**: 2nd-order crossing alone is not enough; you need a **deep network for high-order non-linear crosses** plus **low-order memorization** to reliably beat LR/FM on industrial CTR.

> 💼 **实战视角 / Practical angle**
> **中文**：FM/FFM 长期是 CTR 比赛(Criteo/Avazu/KDD Cup)的主力。**FFM 表达力强但参数 ×场数、极易过拟合**，工业界更常用 FM 或其深度变体。记住面试点：*"FM 用因子化让未共现的特征对也能泛化；要不要上 FFM 看数据规模和交叉信号强度。"* 下一节进入深度时代：**Wide & Deep**——把"记忆(wide)"和"泛化(deep)"显式拆开。
> **English**: FM/FFM long dominated CTR competitions (Criteo/Avazu/KDD Cup). **FFM is expressive but params × #fields and overfits easily**; industry more often uses FM or its deep variants. Interview point: *"FM's factorization lets unseen feature pairs generalize; whether to use FFM depends on data scale and cross-signal strength."* Next, the deep era: **Wide & Deep** — explicitly splitting "memorization (wide)" and "generalization (deep)."

---
### 小结 / Summary
- **中文**：FM = LR + 二阶交叉(隐向量点积参数化)，用 $O(kF)$ 技巧高效计算；MF 是其特例；FFM 加场感知更强但更易过拟合。
- **English**: FM = LR + 2nd-order crossing (latent dot products), computed in $O(kF)$; MF is its special case; FFM adds field-awareness (stronger but overfits more).
- **中文**：合成纯交叉数据上 LR<FM<FFM 一目了然；真实 ID 主导数据上 FM≈LR——增益取决于交叉信号是否真实存在。
- **English**: On synthetic pure-cross data LR<FM<FFM is clear; on real ID-dominated data FM≈LR — the gain depends on whether cross signal truly exists.
- **中文**：评估 CTR 用 AUC/LogLoss；这正是 DeepFM/DCN 要解决的"高阶交叉"问题的起点。
- **English**: Evaluate CTR by AUC/LogLoss; this is the starting point for the "high-order crossing" that DeepFM/DCN address.
